In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from qubit_zz_coupling.utils import load_from_file
from qubit_zz_coupling.sample import sample_sine_curve
from qubit_zz_coupling.experiments import Ramsey_experiment_2qtls
from qubit_zz_coupling.config import load_sim_config, build_runtime_from_config

In [ ]:
# Load configuration
cfg = load_sim_config("tls_simulation_config.yml")
runtime = build_runtime_from_config(cfg)

In [ ]:
# Example TLS Coupling over time (hr vs Ghz)
sample_args = runtime["sample_args"]        # list of positional args for sample_sine_curve
sample_plot = runtime["sample_plot"]
x, y = sample_sine_curve(*sample_args, plot=sample_plot)

## Run the simulation set up in the tls_simulation_config.yml

In [ ]:
# All Two-qubit simultaneous-Ramsey with ZZ coupling + tls
# ---------------------------------------------------------------
params = runtime["experiment_params"]
name_str = runtime["name_str"]

filelist = []

for j in range(runtime["loop"]):
    j_str = f"n{j}"
    x,y = sample_sine_curve(*sample_args, plot=False) # refresh 
    y = np.clip(y, 0, None)

    expA_list = []
    expB_list = []
    counter = 0 

    for i in y:
        if counter%10==0:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            print(f'point: {counter} time: {timestamp}')
        counter+=1
    
        params["chi_MHz_T"] = i # vary tls coupling 

        exp = Ramsey_experiment_2qtls(params)
        expA, expB = exp.run()
        expA_list.append(expA)
        expB_list.append(expB)
        
    # write to file
    # Create a timestamp string
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Create filename
    filename = f"{name_str}_{j_str}_{timestamp}.npz"
    
    # Save the lists to a compressed npz file
    np.savez(filename, expA_list=expA_list, expB_list=expB_list, x=x, y=y, t_ns=params["t_ns"])
    
    filelist.append(filename)
    print(f"Saved results to {filename}")

np.savez(f"{name_str}_files_{timestamp}.npz",filelist=filelist)
print(f"{name_str}_files_{timestamp}.npz")

## Load and Review Sim

In [ ]:
# Load the file list
listfile = f"{name_str}_files_{timestamp}.npz"
flist = np.load(listfile)['filelist']

# List of data elements to return from file
fldlist = ["expA_list","expB_list","x","y","t_ns"]

# Arrays to collect all Runs
expA_list = [] 
expB_list = [] 
x = []  
y = []
t_ns = []

# Load all files
for fname in flist:
    print(fname)
    _a, _b, _x, _y, _t = load_from_file(fname, fldlist)
    
    expA_list.append(_a)
    expB_list.append(_b)
    x.append(_x)
    y.append(_y)
    t_ns.append(_t)
    
# Concatenate to a single array for each
expA_list_flat = np.concatenate(expA_list)
expB_list_flat = np.concatenate(expB_list)
x_flat = np.concatenate(x)
y_flat = np.concatenate(y)
t_ns = t_ns[0] # really only need one

In [ ]:
# Resample time points if desired 
s=1 # is no resampling
if s!=1 :
    expA_list_s = [_ea[::s] for _ea in expA_list_flat]
    expB_list_s = [_eb[::s] for _eb in expB_list_flat]
else :
    expA_list_s = expA_list_flat
    expB_list_s = expB_list_flat

In [ ]:
# Plot one example
expA = expA_list_s[1]
expB = expB_list_s[1]

t_ns = np.linspace(0, 100000, len(expA)) # resampled times

# Plot
plt.figure(figsize=(10,4))
plt.subplot(121); plt.plot(t_ns/1000,expA,'.-'); plt.title('Qubit 5'); plt.xlabel('delay (µs)'); plt.ylabel('P(e)')
plt.subplot(122); plt.plot(t_ns/1000,expB,'.-'); plt.title('Qubit 3'); plt.xlabel('delay (µs)')
plt.tight_layout(); plt.show()